# 04 – Modeling & Evaluation
**Owner:** Danniecia  
**Goal:** Compare a baseline and four models with cross-validated tuning, then recommend a final model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
FIGURES = Path("../reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv(PROCESSED / "games_processed.csv")
FEATURES = [c for c in df.columns if c.endswith("_diff")]  # update with final set from notebook 03
X, y = df[FEATURES], df["HOME_TEAM_WINS"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)

## Models and hyperparameter grids

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models = {
    "Baseline (majority)": (DummyClassifier(strategy="most_frequent"), {}),
    "Logistic Regression": (Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))]),
                            {"clf__C": [0.01, 0.1, 1, 10]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=RANDOM_STATE),
                      {"max_depth": [3, 5, 7, 10], "min_samples_leaf": [1, 20, 50]}),
    "kNN": (Pipeline([("scale", StandardScaler()), ("clf", KNeighborsClassifier())]),
            {"clf__n_neighbors": [5, 15, 31, 51], "clf__weights": ["uniform", "distance"]}),
    "Random Forest": (RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
                      {"n_estimators": [200, 400], "max_depth": [5, 10, None], "min_samples_leaf": [1, 10]}),
}

## Train, tune, and evaluate

In [ ]:
results, fitted = [], {}
for name, (model, grid) in models.items():
    search = GridSearchCV(model, grid, cv=cv, scoring="roc_auc", n_jobs=-1) if grid else model
    search.fit(X_train, y_train)
    best = search.best_estimator_ if grid else search
    fitted[name] = best
    pred = best.predict(X_test)
    proba = best.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Best params": search.best_params_ if grid else "-",
        "CV ROC-AUC": round(search.best_score_, 3) if grid else "-",
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    })
results_df = pd.DataFrame(results).set_index("Model").round(3)
results_df

## Stability check: CV score spread for each tuned model

In [ ]:
from sklearn.model_selection import cross_val_score
for name, m in fitted.items():
    s = cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name:22s} mean={s.mean():.3f}  std={s.std():.3f}")

## Final model

In [ ]:
FINAL = "Logistic Regression"  # update after comparing performance, interpretability, and stability
ConfusionMatrixDisplay.from_estimator(fitted[FINAL], X_test, y_test, cmap="Greys")
plt.show()

## Recommendation
Explain why the final model was chosen (performance, interpretability, stability) and what the results say about which statistics drive wins.
- 